# Qwen2.5-VL-7B Unsloth BNB4 Model Cache

This notebook downloads the model weights once and publishes them as a private Kaggle dataset for reuse.

In [ ]:
import glob
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime

MODEL_REPO = "unsloth/qwen2.5-vl-7b-instruct-unsloth-bnb-4bit"
DATASET_SLUG = "qwen2-5-vl-7b-unsloth-bnb4-cache"
DATASET_TITLE = "Qwen2.5-VL-7B Unsloth BNB4 Cache"
WORKDIR = "/kaggle/working/model_cache"
MODEL_DIR = os.path.join(WORKDIR, "model")
DATASET_DIR = os.path.join(WORKDIR, "dataset")
LOG_PATH = os.path.join(WORKDIR, "progress.txt")

os.makedirs(WORKDIR, exist_ok=True)

def log(message: str) -> None:
    timestamp = datetime.utcnow().isoformat()
    line = f"{timestamp} {message}"
    print(line, flush=True)
    with open(LOG_PATH, "a", encoding="utf-8") as handle:
        handle.write(line + "\n")

log("Notebook start")


In [ ]:
log("Installing packages")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "modelscope==1.35.4", "kaggle"],
    check=True,
)
log("Packages installed")


In [ ]:
log("Configuring Kaggle API")
candidate_keys = glob.glob("/kaggle/input/**/kaggle.json", recursive=True)
if not candidate_keys:
    raise FileNotFoundError("Could not find kaggle.json in /kaggle/input. Attach your kaggle-api-key dataset.")
kaggle_json = candidate_keys[0]
os.makedirs("/root/.kaggle", exist_ok=True)
shutil.copy(kaggle_json, "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)
log(f"Using Kaggle API key from {kaggle_json}")


In [ ]:
from modelscope.hub.snapshot_download import snapshot_download

os.makedirs(MODEL_DIR, exist_ok=True)
log("Starting model download")
snapshot_download(repo_id=MODEL_REPO, local_dir=MODEL_DIR)
log("Model download complete")


In [ ]:
log("Preparing dataset directory")
if os.path.exists(DATASET_DIR):
    shutil.rmtree(DATASET_DIR)
os.makedirs(DATASET_DIR, exist_ok=True)
shutil.copytree(MODEL_DIR, os.path.join(DATASET_DIR, "model"), dirs_exist_ok=True)

metadata = {
    "title": DATASET_TITLE,
    "id": f"kgaero/{DATASET_SLUG}",
    "licenses": [{"name": "CC0-1.0"}],
    "isPrivate": True,
}
with open(os.path.join(DATASET_DIR, "dataset-metadata.json"), "w", encoding="utf-8") as handle:
    json.dump(metadata, handle, indent=2)
log("Dataset metadata written")


In [ ]:
log("Uploading dataset to Kaggle")
create_cmd = [
    "kaggle",
    "datasets",
    "create",
    "-p",
    DATASET_DIR,
    "--dir-mode",
    "zip",
]
result = subprocess.run(create_cmd, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    log("Create failed, attempting version update")
    version_cmd = [
        "kaggle",
        "datasets",
        "version",
        "-p",
        DATASET_DIR,
        "-m",
        "Update model cache",
        "--dir-mode",
        "zip",
    ]
    result = subprocess.run(version_cmd, check=True, capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)
log("Dataset upload complete")
